# 2.2 Tools and structured outputs

Follows [02-tools.md](../02-tools.md). One cell per demo: the schema the model sees, a
validation retry, a timeout that pays twice, a cache hit, and a typed plan.

`MODE = "fake"`: scripted model, placeholder images, no network. `"fake-gpu"`: real LLM on
Token Factory for the validation and plan demos; placeholder images throughout.

In [ ]:
MODE = "fake"  # "fake" or "fake-gpu"

In [ ]:
import sys
from pathlib import Path

MODULE = "02-multi-agent-media-workflows"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
CODE = next(p for p in candidates if (p / "tools.py").exists())
sys.path.insert(0, str(CODE))
ASSETS = CODE.parent / "assets"

In [ ]:
from pydantic_ai import Agent, Tool

from fake import PlaceholderImages, block_real_models, call, scripted_model
from models import Brief
from tools import Store, ToolDeps, generate_image, token_factory_model
from tracing import Tracer, new_run_id, run_traced

if MODE == "fake":
    block_real_models()
brief = Brief.from_yaml(ASSETS / "brief.yaml")


def new_deps(sleep_s: float = 0.2) -> ToolDeps:
    """A fresh run folder, trace and placeholder image backend."""
    tracer = Tracer(new_run_id())
    return ToolDeps(tracer, PlaceholderImages(sleep_s), Store(tracer.folder))


def show(deps: ToolDeps) -> None:
    """Print the trace of a demo, one line per event."""
    for e in deps.tracer.events():
        print(
            f"{e.kind:14} {e.name:16} cache_hit={e.cache_hit!s:5} t_gen_s={e.t_gen_s} cost={e.cost_usd}"
        )
    print("GPU requests received:", deps.images.requests)

## The tool signature is the schema

`generate_image` in `tools.py` has typed arguments. PydanticAI turns them, and the docstring,
into the JSON schema the model receives every turn.

In [ ]:
import json

from pydantic_ai.messages import ModelResponse, TextPart
from pydantic_ai.models.function import FunctionModel


def print_schema(messages, info):
    print(json.dumps(info.function_tools[0].parameters_json_schema, indent=1))
    return ModelResponse(parts=[TextPart("ok")])


await Agent(FunctionModel(print_schema), deps_type=ToolDeps, tools=[generate_image]).run("show")

## Validation retry

An invalid `size` fails validation before any GPU call. The error goes back to the model,
which fixes the call. The trace shows two model requests and one GPU request.

In [ ]:
deps = new_deps()
model = (
    scripted_model(
        call("generate_image", prompt=brief.shot, size="1000x1000"),  # not an allowed size
        call("generate_image", prompt=brief.shot, size="832x480"),
        "Keyframe ready.",
    )
    if MODE == "fake"
    else token_factory_model()
)
agent = Agent(model, deps_type=ToolDeps, tools=[generate_image])
await run_traced(
    agent,
    f"Make the keyframe for: {brief.shot}. Use size 1000x1000.",
    tracer=deps.tracer,
    deps=deps,
)
show(deps)

## Tool timeouts

A timeout below the generation time is a retryable failure in PydanticAI. The client gives up,
the GPU keeps working, and the retry sends a second request. Same tool, timeout above it: one.

In [ ]:
from pydantic_ai import UnexpectedModelBehavior

for timeout in (1, 10):
    deps = new_deps(sleep_s=3)  # this "GPU" takes 3 s per image
    agent = Agent(
        scripted_model(*[call("generate_image", prompt=brief.shot)] * 2, "Keyframe ready."),
        deps_type=ToolDeps,
        tools=[Tool(generate_image, timeout=timeout, max_retries=1)],
    )
    try:
        await run_traced(agent, "Make the keyframe.", tracer=deps.tracer, deps=deps)
    except UnexpectedModelBehavior as exc:
        print("Gave up:", exc)
    print(f"timeout={timeout} s -> GPU requests: {deps.images.requests}")

## Cache keys: reuse a completed result

The same spec twice: the second call is a cache hit, `cost_usd` 0, no GPU request.

In [ ]:
deps = new_deps()
same = call("generate_image", prompt=brief.shot, seed=7)
agent = Agent(
    scripted_model(same, call("generate_image", prompt=brief.shot, seed=7), "Done."),
    deps_type=ToolDeps,
    tools=[generate_image],
)
result = await run_traced(agent, "Make the keyframe twice.", tracer=deps.tracer, deps=deps)
show(deps)

**Exercise:** change one byte of an input file and watch the hash change. The video tool (2.4)
hashes the keyframe's bytes, not its name.

In [ ]:
from tools import spec_hash

keyframe = next(deps.store.folder.glob("keyframe-*.png"))
edited = keyframe.with_name("keyframe-edited.png")
data = bytearray(keyframe.read_bytes())
data[-20] ^= 1
edited.write_bytes(data)

for path in (keyframe, edited):
    key = spec_hash(model="video-model", prompt="slow push-in", seed=0, params={}, inputs=[path])
    print(path.name, key[:16])

## Return references, not bytes

What the model got back vs what the run folder holds.

In [ ]:
from tracing import last_tool_return

print("Model sees:", last_tool_return(result.all_messages()))
print("Run folder holds:", keyframe.stat().st_size, "bytes of PNG")

## Structured outputs for LLM steps

The planner returns a typed `Plan`, not free text.

In [ ]:
from models import Plan

model = (
    scripted_model(
        {
            "image_prompt": f"{brief.shot}, photorealistic, warm light",
            "motion_prompt": "slow push-in as the lamp switches on",
        }
    )
    if MODE == "fake"
    else token_factory_model()
)
planner = Agent(
    model,
    output_type=Plan,
    instructions="Turn the ad brief into a keyframe prompt and a motion prompt.",
)
plan = (await planner.run(brief.model_dump_json())).output
plan